# HAR-RV: Modelo Heterogeneo de Volatilidade Realizada

Neste notebook, exploraremos o modelo **HAR-RV** (Heterogeneous Autoregressive
Realized Volatility), proposto por **Corsi (2009)**.

O HAR-RV e uma alternativa simples e poderosa aos modelos GARCH para prever
volatilidade. Ele usa **volatilidade realizada** (calculada a partir de dados
intradiarios) em tres horizontes temporais como regressores.

**Conteudo:**
1. Volatilidade realizada
2. Modelo HAR de Corsi (2009)
3. Heterogeneidade de agentes
4. HAR-RV vs GARCH
5. Extensoes: HAR-RV-J

**Referencias:**
- Corsi, F. (2009). *A simple approximate long-memory model of realized volatility*. Journal of Financial Econometrics, 7(2), 174-196.
- Andersen, T.G., Bollerslev, T., Diebold, F.X. & Labys, P. (2003). *Modeling and forecasting realized volatility*. Econometrica, 71(2), 579-625.
- Baillie, R.T., Bollerslev, T. & Mikkelsen, H.O. (1996). *Fractionally integrated generalized autoregressive conditional heteroskedasticity*. Journal of Econometrics, 74(1), 3-30.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

## 1. Volatilidade realizada

A **volatilidade realizada** (RV) e uma medida ex-post da volatilidade, calculada
a partir de retornos intradiarios de alta frequencia:

$$RV_t^{(d)} = \sum_{i=1}^{M} r_{t,i}^2$$

onde $r_{t,i}$ sao os $M$ retornos intradiarios do dia $t$ (ex: retornos de 5 minutos).

Sob certas condicoes, a RV e um estimador **consistente** da variancia integrada:

$$RV_t^{(d)} \xrightarrow{p} \int_0^1 \sigma_t^2(s) ds \quad \text{quando } M \to \infty$$

O modelo HAR-RV utiliza tres componentes temporais:
- **Diaria**: $RV_t^{(d)}$ — volatilidade do ultimo dia
- **Semanal**: $RV_t^{(w)} = \frac{1}{5} \sum_{i=0}^{4} RV_{t-i}^{(d)}$ — media dos ultimos 5 dias
- **Mensal**: $RV_t^{(m)} = \frac{1}{22} \sum_{i=0}^{21} RV_{t-i}^{(d)}$ — media dos ultimos 22 dias

In [ ]:
# TODO: Visualize rv_daily, rv_weekly, rv_monthly
# Dicas:
# - Carregue: rv_data = pd.read_csv('../data/realized_volatility.csv', parse_dates=['date'], index_col='date')
# - rv_daily = rv_data['rv_daily']
# - rv_weekly = rv_data['rv_weekly']
# - rv_monthly = rv_data['rv_monthly']
# - Use plot_har_components(rv_data.index, rv_daily, rv_weekly, rv_monthly)
# - Observe: rv_monthly e mais suave que rv_daily (efeito da media)
# - Imprima estatisticas descritivas de cada componente

## 2. Modelo HAR de Corsi (2009)

O modelo HAR-RV e uma regressao linear simples:

$$RV_{t+1}^{(d)} = \beta_0 + \beta_d \cdot RV_t^{(d)} + \beta_w \cdot RV_t^{(w)} + \beta_m \cdot RV_t^{(m)} + \varepsilon_{t+1}$$

**Propriedades:**
- Estimacao por **OLS** (minimos quadrados ordinarios) — rapido e simples
- Apesar de ser um modelo AR(22) restrito, captura **memoria longa** na volatilidade
- A inclusao de componentes em multiplos horizontes gera um decaimento lento
  da funcao de autocorrelacao, mimetizando processos de memoria longa

Os coeficientes $\beta_d$, $\beta_w$, $\beta_m$ capturam a importancia relativa
de cada horizonte temporal para prever a volatilidade futura.

In [ ]:
# TODO: Estime HAR-RV por OLS
# Dicas:
# - model_har = HARRV(rv_daily.values)
# - results_har = model_har.fit()
# - print(results_har.summary())
# - Observe: R-squared indica o poder preditivo do modelo
# - Os coeficientes sao significativos (t-values > 2)?

## 3. Heterogeneidade de agentes

A interpretacao economica do HAR-RV baseia-se na **Hipotese de Mercados Heterogeneos**
(Muller et al., 1997):

| Componente | Horizonte | Tipo de agente | Exemplos |
|:---:|:---:|:---:|:---:|
| $\beta_d$ (diario) | 1 dia | Traders de alta frequencia | Day traders, market makers |
| $\beta_w$ (semanal) | 5 dias | Traders de media frequencia | Swing traders, fundos quantitativos |
| $\beta_m$ (mensal) | 22 dias | Investidores de longo prazo | Fundos de pensao, gestoras macro |

A volatilidade observada e resultado da **superposicao** das acoes de agentes
com diferentes horizontes de investimento. Cada grupo reage a informacao
em velocidades diferentes:
- Traders de alta frequencia reagem a choques recentes ($\beta_d$ alto)
- Investidores de longo prazo suavizam informacao ($\beta_m$ significativo)

In [ ]:
# TODO: Compare coeficientes b_d, b_w, b_m
# Dicas:
# - Extraia os coeficientes: results_har.params
# - params[0] = beta_0 (intercepto)
# - params[1] = beta_d (diario)
# - params[2] = beta_w (semanal)
# - params[3] = beta_m (mensal)
# - Faca um grafico de barras dos coeficientes
# - Calcule a contribuicao relativa de cada componente:
#   contrib_d = beta_d * np.std(rv_daily)
#   contrib_w = beta_w * np.std(rv_weekly)
#   contrib_m = beta_m * np.std(rv_monthly)
# - Qual horizonte temporal e mais importante para a previsao?

## 4. HAR-RV vs GARCH

Vamos comparar o poder preditivo do HAR-RV com o GARCH(1,1):

| Aspecto | GARCH(1,1) | HAR-RV |
|:---:|:---:|:---:|
| Dados | Retornos diarios | Volatilidade realizada (intradiarios) |
| Estimacao | MLE (nao-linear) | OLS (linear) |
| Memoria | Curta (exponencial) | Aproxima longa (hiperbolica) |
| Volatilidade | Latente ($\sigma_t$) | Observavel ($RV_t$) |
| Vantagem | Nao requer dados intradiarios | Mais preciso, mais simples |

Para uma comparacao justa, usaremos a **raiz do erro quadratico medio (RMSE)**
das previsoes out-of-sample.

In [ ]:
# TODO: Compare RMSE de previsao HAR-RV vs GARCH(1,1)
# Dicas:
# - Carregue os retornos: sp500 = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
# - returns = sp500['returns']
# - Estime GARCH(1,1): model_garch = GARCH(returns.values, p=1, q=1); res_garch = model_garch.fit()
# - Previsao GARCH: conditional_var = res_garch.conditional_volatility**2
# - Previsao HAR-RV: fitted_rv = results_har.fitted_values
# - Compare com a RV realizada (alinhando as datas!):
#   rmse_garch = np.sqrt(np.mean((rv_actual - conditional_var)**2))
#   rmse_har = np.sqrt(np.mean((rv_actual - fitted_rv)**2))
# - Use plot_realized_vs_conditional() para visualizar
# - Qual modelo preve melhor a volatilidade?

## 5. Extensoes: HAR-RV-J

Uma extensao importante do HAR-RV e a inclusao de um componente de **jumps** (saltos):

$$RV_{t+1}^{(d)} = \beta_0 + \beta_d \cdot RV_t^{(d)} + \beta_w \cdot RV_t^{(w)} + \beta_m \cdot RV_t^{(m)} + \beta_j \cdot J_t + \varepsilon_{t+1}$$

O componente de jumps pode ser estimado como:

$$J_t = \max(RV_t^{(d)} - BV_t, 0)$$

onde $BV_t$ e a **bipower variation** (variacao bipotencia), que e robusta a jumps:

$$BV_t = \frac{\pi}{2} \sum_{i=2}^{M} |r_{t,i}| |r_{t,i-1}|$$

A ideia e que jumps tem um efeito **assimetrico** na volatilidade futura:
dias com jumps grandes podem sinalizar mudancas de regime.

**Nota**: Como nao temos a bipower variation nos dados, vamos aproximar
o componente de jumps usando a diferenca entre a RV diaria e a RV semanal
como proxy para movimentos anomalos.

In [ ]:
# TODO: Estime HAR-RV com componente de jumps
# Dicas:
# - Crie proxy de jumps: jumps = np.maximum(rv_daily.values - rv_weekly.values, 0)
# - Construa a regressao manualmente com numpy/statsmodels:
#   from numpy.linalg import lstsq
#   # Alinhe as series (precisa de 22 lags para rv_monthly)
#   n = len(rv_daily)
#   y = rv_daily.values[22:]  # variavel dependente: RV_{t+1}
#   X = np.column_stack([
#       np.ones(n - 22),
#       rv_daily.values[21:-1],   # RV_d
#       rv_weekly.values[21:-1],  # RV_w
#       rv_monthly.values[21:-1], # RV_m
#       jumps[21:-1]              # J_t
#   ])
#   beta_j, _, _, _ = lstsq(X, y, rcond=None)
# - Compare R^2 do HAR-RV vs HAR-RV-J
# - O coeficiente de jumps (beta_j) e significativo?
# - Jumps melhoram a previsao de volatilidade?